In [ ]:
我有L2 数据如下 table_df
ap1-ap5, bp1-bp5， as1-as5， bs1-bs5, trdp, trdv, DATE, TIME, RIC

请你帮我构造python代码计算如下因子：
1.对于买方，当ap1因为trd 发生变化的时候(在trdp!=0 时候，当前的ap1 相对上一条ap1 上升了), 在其后的x个event中，是否出现了ap1重新回落到之前的水平或者更低的水平，如果有则帮我记录如下值,(如果后面的x个event中有多个则记录累计值)：
    - 记录这个事件为 is_Trd_refill_B， num_Trd_refill_B, 记录新挂单事件新增size为size_Trd_refill_B, 记录档ap1因为trd 发生变化的trd的size  为origin_size_B
2.对于卖方，同理对称的记录1

3.对于买方，当ap1因为trd 发生变化的时候(在trdp!=0 时候，当前的ap1 相对上一条ap1 上升了), 在其后的x个event中，是否出现了bp1达到之前mid的水平或者更高，如果有则帮我记录如下值,(如果后面的x个event中有多个则记录累计值)：
    - 记录这个事件为 is_Trd_upfill_B， num_Trd_upfill_B, 记录新挂单事件新增size为size_Trd_upfill_B, 记录档ap1因为trd 发生变化的trd的size 为origin_size_B
4.对于卖方， 同理对称记录3

5.对于买方，当ap1因为trd 发生变化的时候(在trdp!=0 时候，当前的ap1 相对上一条ap1 上升了), 在其后的x个event中，是否出现了ap1再次上升且是由于cxl产生的(trdp==0)，如果有则帮我记录如下值,(如果后面的x个event中有多个则记录累计值)：
    - 记录这个事件为 is_Trd_upCxl_B， num_Trd_upCxl_B, 记录新撤单事件减少size为size_Trd_upCxl_B, 记录档ap1因为trd 发生变化的trd的size  为origin_size_B
6.对于卖方， 同理对称记录5

7.对于买方，当ap1因为trd 发生变化的时候(在trdp!=0 时候，当前的ap1 相对上一条ap1 上升了), 在其后的x个event中，是否出现了bp1相对原来bp1下降且是由于cxl产生的(trdp==0)，如果有则帮我记录如下值,(如果后面的x个event中有多个则记录累计值)：
    - 记录这个事件为 is_Trd_downCxl_B， num_downCxl_B, 记录新撤单事件减少size为size_Trd_downCxl_B, 记录档ap1因为trd 发生变化的trd的size  为origin_size_B
8.对于卖方， 同理对称记录5





In [ ]:
from pyML.core.ts.operators import ATickTSShift
from pyML import Scalar, Quote
from pyML.core.ts import operators
from pyML.core.ts.operators import ACT

trd_condition = operators.Gt(ACT('B','T', 1), Scalar(0.0))
refill = operators.Mul(operators.Gt(ACT("S",'A',1), Scalar(0.0)), operators.Le(Quote("AP01"), ATickTSShift(Quote("AP01")), 1))
upfill = operators.Mul(operators.Gt(ACT("B",'A',1), Scalar(0.0)), operators.Lt(Quote("BP01"), ATickTSShift(Quote("BP01")), 1))
upCxl  = operators.Mul(operators.Gt(ACT("S",'C',1), Scalar(0.0)), operators.Gt(Quote("AP01"), ATickTSShift(Quote("AP01")), 1))
downCxl = operators.Mul(operators.Gt(ACT("B",'C',1), Scalar(0.0)), operators.Lt(Quote("BP01"), ATickTSShift(Quote("BP01")), 1))


is_Trd_refill_B = operators.Add(operators.Add(operators.Mul(operators.Mul(ATickTSShift(trd_condition, 1), refill), operators.Gt(Quote("TRDP"), Scalar(0.0))),
operators.Mul(operators.Mul(operators.Mul(ATickTSShift(trd_condition, 2), refill), operators.Gt(Quote("TRDP"), Scalar(0.0))), operators.Le(Quote("AP01"), ATickTSShift(Quote("AP01")), 2))),
operators.Mul(operators.Mul(operators.Mul(ATickTSShift(trd_condition, 3), refill), operators.Gt(Quote("TRDP"), Scalar(0.0))), operators.Le(Quote("AP01"), ATickTSShift(Quote("AP01")), 3)))



operators.Mul(ATickTSShift(trd_condition, 1), refill), operators.Gt(Quote("TRDP"), Scalar(0.0))
operators.Mul(ATickTSShift(trd_condition, 2), refill), operators.Gt(Quote("TRDP"), Scalar(0.0)), operators.Le(Quote("AP01"), ATickTSShift(Quote("AP01")), 2),
operators.Mul(ATickTSShift(trd_condition, 3), refill), operators.Gt(Quote("TRDP"), Scalar(0.0)), operators.Le(Quote("AP01"), ATickTSShift(Quote("AP01")), 3),







In [ ]:
import pandas as pd
import numpy as np

# Assuming table_df is a pandas DataFrame containing the L2 data.
# Columns: ap1-ap5, bp1-bp5, as1-as5, bs1-bs5, trdp, trdv, DATE, TIME, RIC

def calculate_factors_optimized(table_df, x):
    # Initialize new columns to store results
    table_df['is_Trd_refill_B'] = 0
    table_df['num_Trd_refill_B'] = 0
    table_df['size_Trd_refill_B'] = 0
    table_df['origin_size_B'] = 0

    table_df['is_Trd_upfill_B'] = 0
    table_df['num_Trd_upfill_B'] = 0
    table_df['size_Trd_upfill_B'] = 0

    table_df['is_Trd_upCxl_B'] = 0
    table_df['num_Trd_upCxl_B'] = 0
    table_df['size_Trd_upCxl_B'] = 0

    table_df['is_Trd_downCxl_B'] = 0
    table_df['num_downCxl_B'] = 0
    table_df['size_Trd_downCxl_B'] = 0

    table_df['is_Trd_refill_S'] = 0
    table_df['num_Trd_refill_S'] = 0
    table_df['size_Trd_refill_S'] = 0
    table_df['origin_size_S'] = 0

    table_df['is_Trd_upfill_S'] = 0
    table_df['num_Trd_upfill_S'] = 0
    table_df['size_Trd_upfill_S'] = 0

    table_df['is_Trd_upCxl_S'] = 0
    table_df['num_Trd_upCxl_S'] = 0
    table_df['size_Trd_upCxl_S'] = 0

    table_df['is_Trd_downCxl_S'] = 0
    table_df['num_downCxl_S'] = 0
    table_df['size_Trd_downCxl_S'] = 0

    # Calculate origin_size_B where ap1 increases due to trade
    table_df['origin_size_B'] = np.where((table_df['trdp'] != 0) & (table_df['ap1'] > table_df['ap1'].shift(1)), table_df['trdv'], 0)
    table_df['origin_size_S'] = np.where((table_df['trdp'] != 0) & (table_df['bp1'] < table_df['bp1'].shift(1)), table_df['trdv'], 0)

    # Create rolling windows for efficient computation
    for offset in range(0, x + 1):
        table_df[f'ap1_shift_{offset}'] = table_df['ap1'].shift(-offset)
        table_df[f'bp1_shift_{offset}'] = table_df['bp1'].shift(-offset)
        table_df[f'as1_shift_{offset}'] = table_df['as1'].shift(-offset)
        table_df[f'bs1_shift_{offset}'] = table_df['bs1'].shift(-offset)
        table_df[f'trdp_shift_{offset}'] = table_df['trdp'].shift(-offset)

    # Vectorized computation for refill events
    trd_condition_B = (table_df['trdp'] != 0) & (table_df['ap1'] > table_df['ap1'].shift(1))
    trd_condition_S = (table_df['trdp'] != 0) & (table_df['bp1'] < table_df['bp1'].shift(1))

    for offset in range(1, x + 1):
        table_df['num_Trd_refill_B'] += np.where(trd_condition_B & (table_df[f'ap1_shift_{offset}'] < table_df['ap1']) & (table_df[f'ap1_shift_{offset}'] < table_df[f'ap1_shift_{offset-1}']), 1, 0)
        table_df['size_Trd_refill_B'] += np.where(trd_condition_B & (table_df[f'ap1_shift_{offset}'] < table_df['ap1']) & (table_df[f'ap1_shift_{offset}'] < table_df[f'ap1_shift_{offset-1}']), table_df[f'as1_shift_{offset}'], 0)
        table_df['num_Trd_refill_S'] += np.where(trd_condition_S & (table_df[f'bp1_shift_{offset}'] > table_df['bp1']) & (table_df[f'bp1_shift_{offset}'] > table_df[f'bp1_shift_{offset-1}']), 1, 0)
        table_df['size_Trd_refill_S'] += np.where(trd_condition_S & (table_df[f'bp1_shift_{offset}'] > table_df['bp1']) & (table_df[f'bp1_shift_{offset}'] > table_df[f'bp1_shift_{offset-1}']), table_df[f'bs1_shift_{offset}'], 0)

    table_df['is_Trd_refill_B'] = (table_df['num_Trd_refill_B'] > 1).astype(int)
    table_df['is_Trd_refill_S'] = (table_df['num_Trd_refill_S'] > 1).astype(int)

    for offset in range(1, x + 1):
        table_df['num_Trd_upfill_B'] += np.where(trd_condition_B & (table_df[f'bp1_shift_{offset}'] > (table_df['ap1'].shift(1) + table_df['bp1'].shift(1)) / 2) & (table_df[f'bp1_shift_{offset}'] > table_df[f'bp1_shift_{offset-1}']), 1, 0)
        table_df['size_Trd_upfill_B'] += np.where(trd_condition_B & (table_df[f'bp1_shift_{offset}'] > (table_df['ap1'].shift(1) + table_df['bp1'].shift(1)) / 2) & (table_df[f'bp1_shift_{offset}'] > table_df[f'bp1_shift_{offset-1}']), table_df[f'bs1_shift_{offset}'], 0)
        table_df['num_Trd_upfill_S'] += np.where(trd_condition_S & (table_df[f'ap1_shift_{offset}'] < (table_df['ap1'].shift(1) + table_df['bp1'].shift(1)) / 2) & (table_df[f'ap1_shift_{offset}'] < table_df[f'ap1_shift_{offset-1}']), 1, 0)
        table_df['size_Trd_upfill_S'] += np.where(trd_condition_S & (table_df[f'ap1_shift_{offset}'] < (table_df['ap1'].shift(1) + table_df['bp1'].shift(1)) / 2) & (table_df[f'ap1_shift_{offset}'] < table_df[f'ap1_shift_{offset-1}']), table_df[f'as1_shift_{offset}'], 0)

    table_df['is_Trd_upfill_B'] = (table_df['num_Trd_upfill_B'] > 1).astype(int)
    table_df['is_Trd_upfill_S'] = (table_df['num_Trd_upfill_S'] > 1).astype(int)

    # Vectorized computation for upCxl events
    for offset in range(1, x + 1):
        table_df['num_Trd_upCxl_B'] += np.where(trd_condition_B & (table_df[f'ap1_shift_{offset}'] > table_df['ap1']) & (table_df[f'ap1_shift_{offset}'] > table_df[f'ap1_shift_{offset-1}']) & (table_df[f'trdp_shift_{offset}'] == 0), 1, 0)
        table_df['size_Trd_upCxl_B'] += np.where(trd_condition_B & (table_df[f'ap1_shift_{offset}'] > table_df['ap1']) & (table_df[f'ap1_shift_{offset}'] > table_df[f'ap1_shift_{offset-1}']) & (table_df[f'trdp_shift_{offset}'] == 0), table_df[f'as1_shift_{offset}'], 0)
        table_df['num_Trd_upCxl_S'] += np.where(trd_condition_S & (table_df[f'bp1_shift_{offset}'] < table_df['bp1']) & (table_df[f'bp1_shift_{offset}'] < table_df[f'bp1_shift_{offset-1}']) & (table_df[f'trdp_shift_{offset}'] == 0), 1, 0)
        table_df['size_Trd_upCxl_S'] += np.where(trd_condition_S & (table_df[f'bp1_shift_{offset}'] < table_df['bp1']) & (table_df[f'bp1_shift_{offset}'] < table_df[f'bp1_shift_{offset-1}']) & (table_df[f'trdp_shift_{offset}'] == 0), table_df[f'bs1_shift_{offset}'], 0)

    table_df['is_Trd_upCxl_B'] = (table_df['num_Trd_upCxl_B'] > 1).astype(int)
    table_df['is_Trd_upCxl_S'] = (table_df['num_Trd_upCxl_S'] > 1).astype(int)

    # Vectorized computation for downCxl events
    for offset in range(1, x + 1):
        table_df['num_downCxl_B'] += np.where(trd_condition_B & (table_df[f'bp1_shift_{offset}'] < table_df['bp1']) & (table_df[f'bp1_shift_{offset}'] < table_df[f'bp1_shift_{offset-1}']) & (table_df[f'trdp_shift_{offset}'] == 0), 1, 0)
        table_df['size_Trd_downCxl_B'] += np.where(trd_condition_B & (table_df[f'bp1_shift_{offset}'] < table_df['bp1']) & (table_df[f'bp1_shift_{offset}'] < table_df[f'bp1_shift_{offset-1}']) & (table_df[f'trdp_shift_{offset}'] == 0), table_df[f'bs1_shift_{offset}'], 0)
        table_df['num_downCxl_S'] += np.where(trd_condition_S & (table_df[f'ap1_shift_{offset}'] > table_df['ap1']) & (table_df[f'ap1_shift_{offset}'] > table_df[f'ap1_shift_{offset-1}']) & (table_df[f'trdp_shift_{offset}'] == 0), 1, 0)
        table_df['size_Trd_downCxl_S'] += np.where(trd_condition_S & (table_df[f'ap1_shift_{offset}'] > table_df['ap1']) & (table_df[f'ap1_shift_{offset}'] > table_df[f'ap1_shift_{offset-1}']) & (table_df[f'trdp_shift_{offset}'] == 0), table_df[f'as1_shift_{offset}'], 0)

    table_df['is_downCxl_B'] = (table_df['num_downCxl_B'] > 1).astype(int)
    table_df['is_downCxl_S'] = (table_df['num_downCxl_S'] > 1).astype(int)

    # Drop temporary columns
    table_df.drop(columns=[col for col in table_df.columns if 'shift_' in col], inplace=True)

    return table_df

# Example usage:
table_df = pd.DataFrame({
    'ap1': [1, 2, 3, 2, 1],
    'bp1': [1, 1, 2, 3, 2],
    'as1': [10, 20, 30, 40, 50],
    'bs1': [15, 25, 35, 45, 55],
    'trdp': [0, 1, 0, 1, 0],
    'trdv': [0, 100, 0, 200, 0],
    'DATE': ['2026-02-14'] * 5,
    'TIME': ['09:30:00', '09:30:01', '09:30:02', '09:30:03', '09:30:04'],
    'RIC': ['0001.HK'] * 5
})
x = 3
result_df = calculate_factors_optimized(table_df, x)
print(result_df)

请你帮我构造python代码计算如下因子：
1.对于买方，当bp1因为add 发生变化的时候(当前的bp1 相对上一条bp1 上升了), 在其后的x个event中，是否出现了bp1重新回落到之前的水平或者更低的水平且由于trd_from_S产生(trdp!=0)，如果有则帮我记录如下值,(如果后面的x个event中有多个则记录累计值)：
    - 记录这个事件为 is_Addnew_fill_B， num_Addnew_fill_B, 记录新trd事件size为size_Addnew_fill_B, 记录档ap1因为add 发生变化的Add的size  为origadd_size_B
2.对于卖方，同理对称的记录1

3.对于买方，当bp1因为add 发生变化的时候(当前的bp1 相对上一条bp1 上升了), 在其后的x个event中，是否出现了该档位size变大(有人在当前Newlevel后面继续挂单)
    - 记录这个事件为 is_Addnew_add_B， num_Addnew_add_B, 记录新挂单事件新增size为size_Addnew_add_B, 记录档ap1因为trd 发生变化的trd的size 为origin_size_B
4.对于卖方， 同理对称记录3





In [ ]:
def calculate_additional_factors(table_df, x):
    # Initialize new columns for additional factors
    table_df['is_Addnew_fill_B'] = 0
    table_df['num_Addnew_fill_B'] = 0
    table_df['size_Addnew_fill_B'] = 0
    table_df['origadd_size_B'] = 0

    table_df['is_Addnew_add_B'] = 0
    table_df['num_Addnew_add_B'] = 0
    table_df['size_Addnew_add_B'] = 0

    table_df['is_Addnew_fill_S'] = 0
    table_df['num_Addnew_fill_S'] = 0
    table_df['size_Addnew_fill_S'] = 0
    table_df['origadd_size_S'] = 0

    table_df['is_Addnew_add_S'] = 0
    table_df['num_Addnew_add_S'] = 0
    table_df['size_Addnew_add_S'] = 0

    # Create rolling windows for efficient computation
    for offset in range(1, x + 1):
        table_df[f'ap1_shift_{offset}'] = table_df['ap1'].shift(-offset)
        table_df[f'bp1_shift_{offset}'] = table_df['bp1'].shift(-offset)
        table_df[f'as1_shift_{offset}'] = table_df['as1'].shift(-offset)
        table_df[f'bs1_shift_{offset}'] = table_df['bs1'].shift(-offset)
        table_df[f'trdp_shift_{offset}'] = table_df['trdp'].shift(-offset)

    # Vectorized computation for Addnew_fill_B
    add_condition_B = (table_df['bp1'] > table_df['bp1'].shift(1))
    for offset in range(1, x + 1):
        valid_shift = table_df[f'bp1_shift_{offset}'] != table_df[f'bp1_shift_{offset - 1}']
        table_df['num_Addnew_fill_B'] += np.where(add_condition_B & valid_shift & (table_df[f'bp1_shift_{offset}'] <= table_df['bp1'].shift(1)) & (table_df[f'trdp_shift_{offset}'] != 0), 1, 0)
        table_df['size_Addnew_fill_B'] += np.where(add_condition_B & valid_shift & (table_df[f'bp1_shift_{offset}'] <= table_df['bp1'].shift(1)) & (table_df[f'trdp_shift_{offset}'] != 0), table_df[f'bs1_shift_{offset}'], 0)
    table_df['is_Addnew_fill_B'] = (table_df['num_Addnew_fill_B'] > 1).astype(int)

    # Vectorized computation for Addnew_add_B
    for offset in range(1, x + 1):
        valid_shift = table_df[f'bp1_shift_{offset}'] != table_df[f'bp1_shift_{offset - 1}']
        table_df['num_Addnew_add_B'] += np.where(add_condition_B & valid_shift & (table_df[f'bp1_shift_{offset}'] > table_df['bp1']) & (table_df[f'as1_shift_{offset}'] > 0), 1, 0)
        table_df['size_Addnew_add_B'] += np.where(add_condition_B & valid_shift & (table_df[f'bp1_shift_{offset}'] > table_df['bp1']) & (table_df[f'as1_shift_{offset}'] > 0), table_df[f'as1_shift_{offset}'], 0)
    table_df['is_Addnew_add_B'] = (table_df['num_Addnew_add_B'] > 1).astype(int)

    # Vectorized computation for Addnew_fill_S
    add_condition_S = (table_df['ap1'] < table_df['ap1'].shift(1))
    for offset in range(1, x + 1):
        valid_shift = table_df[f'ap1_shift_{offset}'] != table_df[f'ap1_shift_{offset - 1}']
        table_df['num_Addnew_fill_S'] += np.where(add_condition_S & valid_shift & (table_df[f'ap1_shift_{offset}'] >= table_df['ap1'].shift(1)) & (table_df[f'trdp_shift_{offset}'] != 0), 1, 0)
        table_df['size_Addnew_fill_S'] += np.where(add_condition_S & valid_shift & (table_df[f'ap1_shift_{offset}'] >= table_df['ap1'].shift(1)) & (table_df[f'trdp_shift_{offset}'] != 0), table_df[f'as1_shift_{offset}'], 0)
    table_df['is_Addnew_fill_S'] = (table_df['num_Addnew_fill_S'] > 1).astype(int)

    # Vectorized computation for Addnew_add_S
    for offset in range(1, x + 1):
        valid_shift = table_df[f'ap1_shift_{offset}'] != table_df[f'ap1_shift_{offset - 1}']
        table_df['num_Addnew_add_S'] += np.where(add_condition_S & valid_shift & (table_df[f'ap1_shift_{offset}'] < table_df['ap1']) & (table_df[f'bs1_shift_{offset}'] > 0), 1, 0)
        table_df['size_Addnew_add_S'] += np.where(add_condition_S & valid_shift & (table_df[f'ap1_shift_{offset}'] < table_df['ap1']) & (table_df[f'bs1_shift_{offset}'] > 0), table_df[f'bs1_shift_{offset}'], 0)
    table_df['is_Addnew_add_S'] = (table_df['num_Addnew_add_S'] > 1).astype(int)

    # Drop temporary columns
    table_df.drop(columns=[col for col in table_df.columns if 'shift_' in col], inplace=True)

    return table_df

# Example usage:
x = 3
result_df = calculate_additional_factors(table_df, x)
print(result_df)

In [ ]:
def calculate_cxl_factors(table_df, x):
    # Initialize new columns for Cxl factors
    table_df['is_Cxl_sameAdd_B'] = 0
    table_df['num_Cxl_sameAdd_B'] = 0
    table_df['size_Cxl_sameAdd_B'] = 0
    table_df['origcxl_size_B'] = 0

    table_df['is_Cxl_crossAdd_B'] = 0
    table_df['num_Cxl_crossAdd_B'] = 0
    table_df['size_Cxl_crossAdd_B'] = 0

    table_df['is_Cxl_sameAdd_S'] = 0
    table_df['num_Cxl_sameAdd_S'] = 0
    table_df['size_Cxl_sameAdd_S'] = 0
    table_df['origcxl_size_S'] = 0

    table_df['is_Cxl_crossAdd_S'] = 0
    table_df['num_Cxl_crossAdd_S'] = 0
    table_df['size_Cxl_crossAdd_S'] = 0

    # Create rolling windows for efficient computation
    for offset in range(1, x + 1):
        table_df[f'ap1_shift_{offset}'] = table_df['ap1'].shift(-offset)
        table_df[f'bp1_shift_{offset}'] = table_df['bp1'].shift(-offset)
        table_df[f'as1_shift_{offset}'] = table_df['as1'].shift(-offset)
        table_df[f'bs1_shift_{offset}'] = table_df['bs1'].shift(-offset)
        table_df[f'trdp_shift_{offset}'] = table_df['trdp'].shift(-offset)

    # Vectorized computation for Cxl_sameAdd_B
    cxl_condition_B = (table_df['trdp'] == 0) & (table_df['bp1'] < table_df['bp1'].shift(1))
    for offset in range(1, x + 1):
        valid_shift = table_df[f'bp1_shift_{offset}'] != table_df[f'bp1_shift_{offset - 1}']
        table_df['num_Cxl_sameAdd_B'] += np.where(cxl_condition_B & valid_shift & (table_df[f'bp1_shift_{offset}'] >= table_df['bp1'].shift(1)), 1, 0)
        table_df['size_Cxl_sameAdd_B'] += np.where(cxl_condition_B & valid_shift & (table_df[f'bp1_shift_{offset}'] >= table_df['bp1'].shift(1)), table_df[f'bs1_shift_{offset}'], 0)
    table_df['is_Cxl_sameAdd_B'] = (table_df['num_Cxl_sameAdd_B'] > 1).astype(int)

    # Vectorized computation for Cxl_crossAdd_B
    for offset in range(1, x + 1):
        valid_shift = table_df[f'ap1_shift_{offset}'] != table_df[f'ap1_shift_{offset - 1}']
        table_df['num_Cxl_crossAdd_B'] += np.where(cxl_condition_B & valid_shift & (table_df[f'ap1_shift_{offset}'] <= (table_df['ap1'].shift(1) + table_df['bp1'].shift(1)) / 2), 1, 0)
        table_df['size_Cxl_crossAdd_B'] += np.where(cxl_condition_B & valid_shift & (table_df[f'ap1_shift_{offset}'] <= (table_df['ap1'].shift(1) + table_df['bp1'].shift(1)) / 2), table_df[f'as1_shift_{offset}'], 0)
    table_df['is_Cxl_crossAdd_B'] = (table_df['num_Cxl_crossAdd_B'] > 1).astype(int)

    # Vectorized computation for Cxl_sameAdd_S
    cxl_condition_S = (table_df['trdp'] == 0) & (table_df['ap1'] > table_df['ap1'].shift(1))
    for offset in range(1, x + 1):
        valid_shift = table_df[f'ap1_shift_{offset}'] != table_df[f'ap1_shift_{offset - 1}']
        table_df['num_Cxl_sameAdd_S'] += np.where(cxl_condition_S & valid_shift & (table_df[f'ap1_shift_{offset}'] <= table_df['ap1'].shift(1)), 1, 0)
        table_df['size_Cxl_sameAdd_S'] += np.where(cxl_condition_S & valid_shift & (table_df[f'ap1_shift_{offset}'] <= table_df['ap1'].shift(1)), table_df[f'as1_shift_{offset}'], 0)
    table_df['is_Cxl_sameAdd_S'] = (table_df['num_Cxl_sameAdd_S'] > 1).astype(int)

    # Vectorized computation for Cxl_crossAdd_S
    for offset in range(1, x + 1):
        valid_shift = table_df[f'bp1_shift_{offset}'] != table_df[f'bp1_shift_{offset - 1}']
        table_df['num_Cxl_crossAdd_S'] += np.where(cxl_condition_S & valid_shift & (table_df[f'bp1_shift_{offset}'] >= (table_df['ap1'].shift(1) + table_df['bp1'].shift(1)) / 2), 1, 0)
        table_df['size_Cxl_crossAdd_S'] += np.where(cxl_condition_S & valid_shift & (table_df[f'bp1_shift_{offset}'] >= (table_df['ap1'].shift(1) + table_df['bp1'].shift(1)) / 2), table_df[f'bs1_shift_{offset}'], 0)
    table_df['is_Cxl_crossAdd_S'] = (table_df['num_Cxl_crossAdd_S'] > 1).astype(int)

    # Drop temporary columns
    table_df.drop(columns=[col for col in table_df.columns if 'shift_' in col], inplace=True)

    return table_df

# Example usage:
x = 3
result_df = calculate_cxl_factors(table_df, x)
print(result_df)